In [19]:
import pandas as pd
import requests
import json
import numpy as np


In [20]:
def calc_strength(p_wins, p_games,c=5,K=400):
    penalty = c / (p_games + 1)

    p_smoothed = (p_wins+ c)/(p_games + ((2* c) + penalty)) 
    
    strength = K * np.log10(p_smoothed/(1-p_smoothed))
    strength[p_games == 0] = -400
    
    return strength

def bradley_terry(R_i, R_j, base=10, K=400):
    return 1/(1+base**((R_j - R_i)/K))

In [21]:
def simulate_bo3(p_win_array, tau=0.05):
    p = p_win_array.copy()
    picks = []

    def softmax(x):
        e = np.exp(x / tau)
        return e / e.sum()

    def do_ban(worst=True):
        probs = softmax(1 - p) if worst else softmax(p)
        idx = np.random.choice(len(p), p=probs)
        return idx

    def do_pick(best=True):
        probs = softmax(p) if best else softmax(1 - p)
        idx = np.random.choice(len(p), p=probs)
        return idx

    def remove(idx):
        nonlocal p
        p = np.delete(p, idx)

    remove(do_ban(worst=True))
    remove(do_ban(worst=False))
    picks.append(p[do_pick(best=True)]); remove(do_pick(best=True))
    picks.append(p[do_pick(best=False)]); remove(do_pick(best=False))
    remove(do_ban(worst=False))
    remove(do_ban(worst=True))
    picks.append(p[0])
    return np.array(picks)

def mc_sim(p_win_array, N=10000, tau = 0.2):
    outcomes = {2: 0, 1: 0, -1: 0, -2: 0}
    for _ in range(N):
        picks = simulate_bo3(p_win_array, tau=tau)
        score = 0
        for p_win in picks:
            score += 1 if np.random.random() < p_win else -1
            if abs(score) == 2:
                break
        outcomes[score] += 1
    return outcomes

In [22]:
TEAM1_ID = 4494
TEAM2_ID = 13286


In [23]:
team1_stats = json.loads(requests.get(f"https://api.csapi.de/teams/{TEAM1_ID}/stats").text)
team2_stats = json.loads(requests.get(f"https://api.csapi.de/teams/{TEAM2_ID}/stats").text)

team1_stats

[{'id': 0, 'name': 'All', 'n': 18, 'n_wins': 12},
 {'id': 1, 'name': 'Anubis', 'n': 0, 'n_wins': 0},
 {'id': 2, 'name': 'Overpass', 'n': 5, 'n_wins': 5},
 {'id': 3, 'name': 'Ancient', 'n': 4, 'n_wins': 2},
 {'id': 4, 'name': 'Dust2', 'n': 10, 'n_wins': 3},
 {'id': 5, 'name': 'Nuke', 'n': 2, 'n_wins': 0},
 {'id': 6, 'name': 'Mirage', 'n': 8, 'n_wins': 7},
 {'id': 7, 'name': 'Inferno', 'n': 11, 'n_wins': 7}]

In [24]:
rankings = pd.DataFrame(json.loads(requests.get(f"https://api.csapi.de/rankings/").text)['rankings'])

rankings

,id,name,rank,points
0,9565,Vitality,1,2031
1,4494,MOUZ,2,1904
2,11283,Falcons,3,1830
3,8297,FURIA,4,1814
4,12467,PARIVISION,5,1793
...,...,...,...,...
299,13629,HereWeGoAgain,300,409
300,11988,GLORE,301,401
301,13459,ex-Inner Circle,301,401
302,13519,Charrados,303,394


In [25]:
df_t1 = pd.DataFrame(team1_stats)
df_t1['team'] = 1
df_t2 = pd.DataFrame(team2_stats)
df_t2['team'] = 2

df = pd.concat([df_t1,df_t2])

df

,id,name,n,n_wins,team
0,0,All,18,12,1
1,1,Anubis,0,0,1
2,2,Overpass,5,5,1
3,3,Ancient,4,2,1
4,4,Dust2,10,3,1
5,5,Nuke,2,0,1
6,6,Mirage,8,7,1
7,7,Inferno,11,7,1
0,0,All,23,13,2
1,1,Anubis,8,2,2


In [26]:
# df['p'] = df['n_wins'] / df['n']

df['s'] = calc_strength(np.array(df['n_wins']),np.array(df['n']), c = 5)

df

,id,name,n,n_wins,team,s
0,0,All,18,12,1,71.515500
1,1,Anubis,0,0,1,-400.000000
2,2,Overpass,5,5,1,93.633282
3,3,Ancient,4,2,1,-23.196779
4,4,Dust2,10,3,1,-76.895158
5,5,Nuke,2,0,1,-95.552836
6,6,Mirage,8,7,1,105.028698
7,7,Inferno,11,7,1,42.113619
0,0,All,23,13,2,29.276353
1,1,Anubis,8,2,2,-87.077116


In [27]:
df = df[df['id'] != 0]
df

,id,name,n,n_wins,team,s
1,1,Anubis,0,0,1,-400.000000
2,2,Overpass,5,5,1,93.633282
3,3,Ancient,4,2,1,-23.196779
4,4,Dust2,10,3,1,-76.895158
5,5,Nuke,2,0,1,-95.552836
6,6,Mirage,8,7,1,105.028698
7,7,Inferno,11,7,1,42.113619
1,1,Anubis,8,2,2,-87.077116
2,2,Overpass,8,5,2,27.100714
3,3,Ancient,10,5,2,-7.722062


In [28]:
p_win = []

for i in df['id'].unique():

    t1 = df[(df['id'] == i) & (df['team'] == 1)]
    t2 = df[(df['id'] == i) & (df['team'] == 2)]

    p_win.append(bradley_terry(
        t1['s'].item(),
        t2['s'].item()
        )
    )
p_win = np.array(p_win)

p_win


array([0.14168937, 0.59459459, 0.47774481, 0.42307692, 0.33546326,
       0.50019053, 0.9272376 ])

In [29]:
outcomes = mc_sim(p_win)

In [30]:
def outcome_to_df(outcomes):

    lookup ={
        2:'2-0',
        1:'2-1',
        -1:'1-2',
        -2:'0-2'
    }

    outcomes_sub = {lookup[k]: v for k,v in outcomes.items()}


    df_outcomes = pd.DataFrame({
        'outcome': list(outcomes_sub.keys()),
        'n': list(outcomes_sub.values())
    })

    df_outcomes['p'] = df_outcomes['n'] / df_outcomes['n'].sum()

    return df_outcomes

df_outcomes = outcome_to_df(outcomes)

df_outcomes


,outcome,n,p
0,2-0,2032,0.2032
1,2-1,2471,0.2471
2,1-2,2826,0.2826
3,0-2,2671,0.2671


In [31]:
ranking_t1 = rankings[rankings['id'] == TEAM1_ID]['points'].item()
ranking_t2 = rankings[rankings['id'] == TEAM2_ID]['points'].item()

print(ranking_t1)

def strength(ranking, K = 400):
    return K * np.log10(ranking)


1904


In [32]:
alpha = 0.3
p_map_combined = alpha * bradley_terry(ranking_t1, ranking_t2)+ (1-alpha) * p_win


In [33]:
bradley_terry(ranking_t1, ranking_t2)

0.8611409356570433

In [34]:
outcomes = mc_sim(np.array(p_map_combined), N = 10000, tau= 0.2)

df_outcomes = outcome_to_df(outcomes)

df_outcomes

,outcome,n,p
0,2-0,3430,0.3430
1,2-1,2945,0.2945
2,1-2,2004,0.2004
3,0-2,1621,0.1621


In [35]:
outcomes = mc_sim((1-np.array(p_map_combined)), N = 10000, tau= 0.2)

df_outcomes = outcome_to_df(outcomes)

df_outcomes

,outcome,n,p
0,2-0,1499,0.1499
1,2-1,2086,0.2086
2,1-2,2764,0.2764
3,0-2,3651,0.3651
